In [10]:
from sqlalchemy.testing.suite.test_reflection import metadata
!pip install langchain-community langchain-text-splitters langchain-openai langchain-chroma chromadb pypdf

In [38]:
import os
import numpy as np
from numpy import dot
from numpy.linalg import norm
import pandas as pd
from langchain_openai import OpenAIEmbeddings   # langchain.embeddings 경로는 deprecated

embeddings = OpenAIEmbeddings(
    model="text-embedding-qwen3-embedding-8b",
    base_url="http://host.docker.internal:12345/v1",
    api_key="lm-studio",
    check_embedding_ctx_length=False,
    chunk_size=16,
    timeout=60.0,
    max_retries=2,
)

In [36]:
import chromadb
chroma_client = chromadb.HttpClient(host="chromadb", port=8000)
print("heartbeat:", chroma_client.heartbeat())

heartbeat: 1782230072632579095


In [14]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "2024_KB_부동산_보고서_최종.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()
print("청크의 수:", len(pages))

/tmp/ipykernel_83/3413237543.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


청크의 수: 84


In [16]:
import re
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# pages가 다른 셀에서 덮어써졌을 수 있어 항상 새로 로드 (상태 의존 제거)
pages = PyPDFLoader(PDF_PATH).load()
print("원본 페이지:", len(pages))

# 머리글 / 페이지번호 / 차트·표 캡션 soup 제거
HEADER = re.compile(r'2024\s*KB\s*부동산\s*보고서[^\n]*')
PAGENO = re.compile(r'^\s*\d{1,3}\s*$', re.M)
CAPTION = re.compile(r'^\s*(그림|표|자료|주)\s*[ⅠⅡⅢIVX\d][^\n]*$', re.M)

def clean_page(text: str) -> str:
    text = HEADER.sub('', text)
    text = CAPTION.sub('', text)
    text = PAGENO.sub('', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

cleaned = []
for p in pages:
    body = clean_page(p.page_content)
    if len(body) < 50:
        continue
    cleaned.append(Document(page_content=body, metadata=dict(p.metadata)))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", "。", "! ", "? ", " ", ""],
    keep_separator=True,
)
splits = splitter.split_documents(cleaned)
splits = [d for d in splits if len(d.page_content) >= 80]   # 자투리 제거

lens = [len(d.page_content) for d in splits]
print(f"정리 후 페이지 {len(cleaned)} -> 청크 {len(splits)}개 "
      f" | min {min(lens)} / avg {sum(lens)//len(lens)} / max {max(lens)}")

원본 페이지: 84
정리 후 페이지 80 -> 청크 215개  | min 83 / avg 374 / max 499


In [17]:
# 청크 길이 통계
chunk_lengths = [len(chunk.page_content) for chunk in splits]
print('청크의 최대 길이 :', max(chunk_lengths))
print('청크의 최소 길이 :', min(chunk_lengths))
print('청크의 평균 길이 :', sum(chunk_lengths) / len(chunk_lengths))

청크의 최대 길이 : 499
청크의 최소 길이 : 83
청크의 평균 길이 : 374.3906976744186


In [28]:
from langchain_chroma import Chroma
COLLECTION = "kb_realestate_2024"
# (선택) 재실행 시 중복 적재 방지 — 기존 컬렉션 삭제
try:
    chroma_client.delete_collection(COLLECTION)
    print("기존 컬렉션 삭제됨")
except Exception as e:
    print("삭제 건너뜀:", e)

삭제 건너뜀: Collection [kb_realestate_2024] does not exist


In [39]:
vectordb = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    client=chroma_client,
    collection_name=COLLECTION,
    collection_metadata={"hnsw:space":"cosine"}, # 비 OpenAI 임베딩: cosine 권장 # <- 여기 설정 부분에 대해서 조사 필요
)
print('문서의 수:', vectordb._collection.count())

문서의 수: 215


In [43]:
vectordb = Chroma(
    client=chroma_client,
    collection_name=COLLECTION,
    embedding_function=embeddings,
)
print('문서의 수:', vectordb._collection.count())

문서의 수: 215


In [44]:
question = "수도권 주택 매매 전망"
top_docs = vectordb.similarity_search(question, k=2)
for i, doc in enumerate(top_docs, 1):
    print(f"문서 {i}:")
    print(f"내용: {doc.page_content[:150]}...")
    print(f"메타데이터: {doc.metadata}")
    print('--' * 20)

문서 1:
내용: 자료: KB경영연구소 
 
지역별로 수도권과 비수도권 모두 하락 전망이 많았으나, 시장 여건 개선에 대한 기대감이 반영되며 
전문가와 공인중개사의 1/3이 수도권 주택 매매가격 상승을 전망하였다. 비수도권에 대해서는 전문가
들이 조금 더 부정적으로 시장을 예측했다. 전...
메타데이터: {'creationdate': '2024-03-04T15:30:01+09:00', 'author': '손은경', 'page_label': '35', 'source': '2024_KB_부동산_보고서_최종.pdf', 'title': 'Morning Meeting', 'total_pages': 84, 'moddate': '2024-03-04T15:30:01+09:00', 'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'page': 34}
----------------------------------------
문서 2:
내용: Executive Summary 2 
 주택 매매시장 하락 전망 우세, 부동산 투자 선호도 하락 
• 2024년 주택 매매가격 지난해에 이어 올해도 하락 전망 우세, 높은 금리가 가장 큰 부담 
부동산시장 전문가와 공인중개사, 자산관리전문가(PB)를 대상으로 한 설문...
메타데이터: {'creationdate': '2024-03-04T15:30:01+09:00', 'moddate': '2024-03-04T15:30:01+09:00', 'creator': 'Microsoft® Word 2016', 'page_label': '3', 'title': 'Morning Meeting', 'total_pages': 84, 'source': '2024_KB_부동산_보고서_최종.pdf', 'author': '손은경', 'page': 2, 'producer': 'Microsoft® Word 2016'}
----------------------------------------


In [47]:
 def instruction_query(task: str, query: str) -> str:
     return f"Instruct: {task}\nQuery: {query}"

 def search_rows(task, question, k=5):
     q = instruction_query(task, question) if task else question
     hits = vectordb.similarity_search_with_relevance_scores(q, k=k)
     return [(d.metadata.get("page"), round(s, 3), d.page_content.replace("\n", " "))
             for d, s in hits]

 def compare(question, variants, k=3, w=160):
     """같은 질문에 인스트럭션만 바꿔, top 청크가 어떻게 갈리는지 한눈에."""
     print(f"질문: {question!r}\n" + "=" * w)
     seen = {}
     for name, task in variants.items():
         rows = search_rows(task, question, k=k)
         seen[name] = [p for p, _, _ in rows]
         print(f"[{name}] instruct = {task or '(none)'}")
         for p, score, snip in rows:
             print(f"   p{p:>3} {score:.3f} {snip[:w]}")
         print("-" * w)
     print("top 페이지 집합:")
     for name, pages in seen.items():
         print(f"  {name:<10} {pages}")
     print("=" * w + "\n")

In [48]:
compare("전세", {
    "가격통계": "Retrieve passages reporting jense price changes with concrete "
              "percentages, indices, or numeric figures.",
    "리스크"  : "Retrieve passages about jense risks: deposit-return defaults, "
              "reverse-jense, and fraud.",
    "정책 제도": "Retrieve passages about government measures or institutional "
              "changes affecting the jeonse system.",
}, k=3)

질문: '전세'
[가격통계] instruct = Retrieve passages reporting jense price changes with concrete percentages, indices, or numeric figures.
   p 22 0.656 4) 전세 수요 아파트 집중, 입주물량 부족으로 가격 상승 가능성 확대  ■ 2023년 하반기 수도권 아파트를 중심으로 전세가격 회복세를 보였으나 최근 들어 다시 주춤  급격한 금리 인상으로 주택 매매시장뿐 아니라 전세시장도 빠르게 위축되면서 전세가격은 8월 이후  하락세로 전환되었다.
   p  8 0.624 자료: 한국부동산원 자료: 한국부동산원    ■ 전세시장도 하락세가 지속되고 있으나 반등 가능성은 여전히 높은 상황  2023년 주택 전세가격은 전년 대비 5.5% 하락하였다. 매매가격과 마찬가지로 외환위기 직후인  1998년(18.4% 하락) 이후 최대 하락폭이다. 그러나 전세가격은 
   p 37 0.620 전문가와 공인중개사 설문조사 결과, 매매 수요 감소에 따른 전세 수요 증가가 주택 전세가격 상승에  가장 중요한 요인으로 나타났다. 다음으로 신규 입주물량 감소, 전세자금 대출 등 정부 정책 지원에 따 른 수요 증가, 계약 갱신 만료에 따른 이주 수요 증가 순으로 전세가격 상승 요인에 
----------------------------------------------------------------------------------------------------------------------------------------------------------------
[리스크] instruct = Retrieve passages about jense risks: deposit-return defaults, reverse-jense, and fraud.
   p  8 0.570 자료: 한국부동산원 자료: 한국부동산원    ■ 전세시장도 하락세가 지속되고 있으나 반등 가능성은 여전히 높은 상황

In [49]:
compare("지금 집을 사도 될까?", {
    "무주택실수요": "Answer for a first-time homebuyer. Retrieve passages on "
                "affordability, mortage burden (PIR, DSR), and end-user demand.",
    "다주택투자": "Answer for a multi-home investor. Retrieve passages on rental "
                "yield, capital-gain outlook, and promising investment regions.",
    "정책 당국": "Answer from a regulator's view. Retrieve passages on market-"
                "stabilization policy and systemic risk.",
}, k=3)

질문: '지금 집을 사도 될까?'
[무주택실수요] instruct = Answer for a first-time homebuyer. Retrieve passages on affordability, mortage burden (PIR, DSR), and end-user demand.
   p 26 0.647 ■ 고금리에 따른 매수 자금 부담 지속  주택 구매력은 구입하고자 하는 주택가격과 필요한 자금을 조달하는 데 드는 비용에 의해 결정된다.  주택가격이 높으면 그만큼 대출 등을 통해 조달해야 하는 자금이 많아지고 금리가 높으면 원리금 상환 에 필요한 비용 부담이 커진다.  서울 아파트 평
   p 25 0.531 5) 주택 경기에 최대 화두로 부각되는 금리 인하 가능성  ■ 금리 상승 영향으로 주택 거래가 침체되고 시장 내 매물 증가  고가 재화인 주택은 일반적으로 자기 자금과 함께 금융기관 대출 등을 이용하여 구입한다. 통계청에 서 발표한 2023년 가계금융복지조사에 따르면 자가 가구의 46.
   p 27 0.528 ■ 금리 인하에 따른 매수 심리 회복 가능성이 높지만 시장 영향은 제한적일 전망  최근 매수자의 구매력이 개선되고 있지만 매수 심리 회복으로 이어지지 못하고 있다. 기준금리는  2023년 1월 이후 3.5% 수준을 유지하고 있지만 시중은행 주택담보대출 금리는 하락세를 보였다.  2024
----------------------------------------------------------------------------------------------------------------------------------------------------------------
[다주택투자] instruct = Answer for a multi-home investor. Retrieve passages on rental yield, capital-gain outlook, and promising investment regions.
   p 41 0.60

In [50]:
compare("2024년 수도권 주택가격 방향", {
    "하방위험": "Retrieve ONLY passages predicting price declines, downside risks, "
              "or a bearish outlook.",
    "상방요인" : "Retrieve ONLY passages predicting price recovery, upside drivers, "
              "or a bullish outlook.",
}, k=4)

질문: '2024년 수도권 주택가격 방향'
[하방위험] instruct = Retrieve ONLY passages predicting price declines, downside risks, or a bearish outlook.
   p  2 0.700 Executive Summary 2   주택 매매시장 하락 전망 우세, 부동산 투자 선호도 하락  • 2024년 주택 매매가격 지난해에 이어 올해도 하락 전망 우세, 높은 금리가 가장 큰 부담  부동산시장 전문가와 공인중개사, 자산관리전문가(PB)를 대상으로 한 설문 조사 결과, 20
   p 34 0.685 1) 2024년 주택시장 전망  ■ 주택 매매가격, 2024년에도 하락 전망 우세한 가운데 상승 전망 2023년 대비 증가  부동산시장 전문가와 공인중개사, 자산관리전문가(PB)를 대상으로 한 설문조사 결과, 2024년 전국 주 택 매매가격은 하락세가 이어질 것이라는 전망이 우세하였다.
   p 36 0.661 ■ 주택 전세가격, 비수도권 하락 전망이 우세한 가운데 수도권 전망은 엇갈려  2024년 전국 주택 전세가격에 대해 전문가의 53%, 공인중개사의 61%가 하락을 전망하였다. 하락 폭에 대해서는 3% 이하가 될 것이라는 의견이 많았다. 2023년 설문조사 결과와 비교해 하락 전망은  크
   p  5 0.636 Contents    Ⅰ. 2024년 주택시장 진단과 전망  1. 2023년 주택시장 점검과 2024년 전망  2. 주택시장 7대 이슈  1) 역대 최저 수준이 지속되고 있는 주택 거래  2) 주택공급 급격한 감소로 인한 공급 부족 가능성  3) 「노후계획도시 특별법」과 재건축 시장 영
----------------------------------------------------------------------------------------------------------------------------------------------------------------
[상방요인] in

In [52]:
compare("공인중개사의 2024년 집값 전망", {
    "응답수치" : "Retrieve the exact survey response percentages - the share answering "
               "rise, fall, or flat.",
    "전망근거" : "Retrieve the reasons or rationale respondents gave, not the numbers.",
}, k=3)

질문: '인천 부평의 2024년 집값 전망'
[응답수치] instruct = Retrieve the exact survey response percentages - the share answering rise, fall, or flat.
   p 34 0.650 1) 2024년 주택시장 전망  ■ 주택 매매가격, 2024년에도 하락 전망 우세한 가운데 상승 전망 2023년 대비 증가  부동산시장 전문가와 공인중개사, 자산관리전문가(PB)를 대상으로 한 설문조사 결과, 2024년 전국 주 택 매매가격은 하락세가 이어질 것이라는 전망이 우세하였다.
   p  2 0.630 Executive Summary 2   주택 매매시장 하락 전망 우세, 부동산 투자 선호도 하락  • 2024년 주택 매매가격 지난해에 이어 올해도 하락 전망 우세, 높은 금리가 가장 큰 부담  부동산시장 전문가와 공인중개사, 자산관리전문가(PB)를 대상으로 한 설문 조사 결과, 20
   p 36 0.596 자료: KB경영연구소    지역별로 수도권에서는 2024년 주택 전세가격이 상승할 것이라는 의견이 전문가 52%, 공인중개사  53%로 많았으나 하락 의견을 제시한 비율과 큰 차이를 보이지 않아 시장에 대한 전망이 엇갈리는 모 습을 보였다. 비수도권에서는 전문가의 68%와 공인중개사의 
----------------------------------------------------------------------------------------------------------------------------------------------------------------
[전망근거] instruct = Retrieve the reasons or rationale respondents gave, not the numbers.
   p  2 0.591 Executive Summary 2   주택 매매시장 하락 전망 우세, 부동산 투자 선호도 하락  • 2024년 주택 매매가격 지난해에 이어 